In [1]:
from __future__ import annotations

import json
import os
import random
import re
import time
import uuid
from dataclasses import asdict, dataclass, field
from enum import Enum
from pathlib import Path
from typing import Any, Callable, Optional

import numpy as np
import pandas as pd

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "data").exists() and (PROJECT_ROOT.parent / "data").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
DATA_DIR = PROJECT_ROOT / "data"
OUTPUT_DIR = PROJECT_ROOT / "outputs"
OUTPUT_DIR.mkdir(exist_ok=True)

print(f"项目目录: {PROJECT_ROOT.resolve()}")

项目目录: D:\CodeData\Program Coding\Project\Writing_Coach_Agent


In [3]:
class StepStatus(str, Enum):
    PENDING = "pending"
    RUNNING = "running"
    SUCCEEDED = "succeeded"
    FAILED = "failed"


@dataclass
class PlanStep:
    step_id: str
    tool_name: str
    purpose: str
    inputs: dict[str, Any]
    status: StepStatus = StepStatus.PENDING
    result: Any = None
    error: Optional[str] = None


@dataclass
class AgentState:
    task: str
    inputs: dict[str, Any]
    run_id: str = field(default_factory=lambda: uuid.uuid4().hex[:8])
    plan: list[PlanStep] = field(default_factory=list)
    artifacts: dict[str, Any] = field(default_factory=dict)
    trace: list[dict[str, Any]] = field(default_factory=list)
    final_answer: Optional[dict[str, Any]] = None

    def log(self, event: str, **payload: Any) -> None:
        self.trace.append({
            "time": time.strftime("%H:%M:%S"),
            "run_id": self.run_id,
            "event": event,
            **payload,
        })

class ToolRegistry:
    def __init__(self) -> None:
        self._tools: dict[str, Callable[..., Any]] = {}

    def register(self, name: str, func: Callable[..., Any]) -> None:
        if name in self._tools:
            raise ValueError(f"工具已注册: {name}")
        self._tools[name] = func

    def call(self, name: str, **kwargs: Any) -> Any:
        if name not in self._tools:
            raise KeyError(f"未知工具: {name}")
        return self._tools[name](**kwargs)

    @property
    def names(self) -> list[str]:
        return sorted(self._tools)

In [4]:
def clean_text(text: str) -> dict[str, Any]:
    cleaned = re.sub(r"\s+", " ", text).strip()
    return {"cleaned_text": cleaned}


def basic_stats(text: str) -> dict[str, Any]:
    words = re.findall(r"\b[\w']+\b", text)
    sentences = [s.strip() for s in re.split(r"[.!?]+", text) if s.strip()]
    connectors = ["first", "second", "however", "therefore", "because", "for example"]
    found = [c for c in connectors if re.search(rf"\b{re.escape(c)}\b", text, flags=re.I)]
    return {
        "word_count": len(words),
        "sentence_count": len(sentences),
        "avg_sentence_length": round(len(words) / max(len(sentences), 1), 2),
        "connectors": found,
    }


def compose_brief(stats: dict[str, Any]) -> dict[str, Any]:
    strengths, risks = [], []
    if stats["word_count"] >= 60:
        strengths.append("篇幅足以展开基本论证")
    else:
        risks.append("篇幅偏短，论点可能缺少展开")
    if len(stats["connectors"]) >= 2:
        strengths.append("使用了多个篇章连接词")
    else:
        risks.append("段落衔接信号较少")
    return {"strengths": strengths, "risks": risks, "stats": stats}


registry = ToolRegistry()
registry.register("clean_text", clean_text)
registry.register("basic_stats", basic_stats)
registry.register("compose_brief", compose_brief)
registry.names

['basic_stats', 'clean_text', 'compose_brief']

In [5]:
class RulePlanner:
    def create_plan(self, state: AgentState) -> list[PlanStep]:
        return [
            PlanStep("S1", "clean_text", "规范化作文文本", {"text": "$essay"}),
            PlanStep("S2", "basic_stats", "提取可解释的基础统计", {"text": "$cleaned_text"}),
            PlanStep("S3", "compose_brief", "生成第一版诊断摘要", {"stats": "$stats"}),
        ]


def resolve_inputs(spec: dict[str, Any], state: AgentState) -> dict[str, Any]:
    resolved = {}
    for key, value in spec.items():
        if isinstance(value, str) and value.startswith("$"):
            artifact_key = value[1:]
            if artifact_key in state.artifacts:
                resolved[key] = state.artifacts[artifact_key]
            elif artifact_key in state.inputs:
                resolved[key] = state.inputs[artifact_key]
            else:
                raise KeyError(f"找不到计划输入: {artifact_key}")
        else:
            resolved[key] = value
    return resolved

In [6]:
class AgentExecutor:
    def __init__(self, planner: RulePlanner, registry: ToolRegistry) -> None:
        self.planner = planner
        self.registry = registry

    def run(self, task: str, **inputs: Any) -> AgentState:
        state = AgentState(task=task, inputs=inputs, artifacts=dict(inputs))
        state.log("run_started", task=task)
        state.plan = self.planner.create_plan(state)
        state.log("plan_created", steps=[asdict(s) for s in state.plan])

        for step in state.plan:
            step.status = StepStatus.RUNNING
            state.log("tool_started", step_id=step.step_id, tool=step.tool_name)
            try:
                kwargs = resolve_inputs(step.inputs, state)
                result = self.registry.call(step.tool_name, **kwargs)
                step.result = result
                step.status = StepStatus.SUCCEEDED

                if step.tool_name == "clean_text":
                    state.artifacts.update(result)
                elif step.tool_name == "basic_stats":
                    state.artifacts["stats"] = result
                elif step.tool_name == "compose_brief":
                    state.artifacts["brief"] = result
                    state.final_answer = result

                state.log("tool_succeeded", step_id=step.step_id, tool=step.tool_name)
            except Exception as exc:
                step.status = StepStatus.FAILED
                step.error = f"{type(exc).__name__}: {exc}"
                state.log("tool_failed", step_id=step.step_id, tool=step.tool_name, error=step.error)
                break

        state.log("run_finished", success=state.final_answer is not None)
        return state

In [8]:
essays = [json.loads(line) for line in (DATA_DIR / "essays.jsonl").read_text(encoding="utf-8").splitlines()]
sample = essays[0]

agent = AgentExecutor(RulePlanner(), registry)
state = agent.run(
    task="分析作文并给出基础诊断",
    essay=sample["essay"],
    prompt=sample["prompt"],
)

print(json.dumps(state.final_answer, ensure_ascii=False, indent=2))

{
  "strengths": [
    "使用了多个篇章连接词"
  ],
  "risks": [
    "篇幅偏短，论点可能缺少展开"
  ],
  "stats": {
    "word_count": 57,
    "sentence_count": 5,
    "avg_sentence_length": 11.4,
    "connectors": [
      "first",
      "second",
      "therefore",
      "for example"
    ]
  }
}


In [9]:
trace_df = pd.DataFrame(state.trace)
display(trace_df[["time", "event"] + [c for c in ["step_id", "tool", "success"] if c in trace_df.columns]])

assert state.final_answer is not None
assert all(step.status == StepStatus.SUCCEEDED for step in state.plan)
assert len(state.trace) >= 8
print("✅ 第 1 课验收通过：Agent Loop 已完整执行。")

,time,event,step_id,tool,success
0,22:21:09,run_started,NaN,NaN,NaN
1,22:21:09,plan_created,NaN,NaN,NaN
2,22:21:09,tool_started,S1,clean_text,NaN
3,22:21:09,tool_succeeded,S1,clean_text,NaN
4,22:21:09,tool_started,S2,basic_stats,NaN
5,22:21:09,tool_succeeded,S2,basic_stats,NaN
6,22:21:09,tool_started,S3,compose_brief,NaN
7,22:21:09,tool_succeeded,S3,compose_brief,NaN
8,22:21:09,run_finished,NaN,NaN,True


✅ 第 1 课验收通过：Agent Loop 已完整执行。
